In [684]:
import os
import pandas as pd
import gc
import numpy as np
from pathlib import Path
from category_encoders import CountEncoder
from sklearn.compose import make_column_selector
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

In [685]:
BASE_DIR = Path.cwd().parent
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_FILE = os.path.join(DATA_DIR, "train_raw.csv")
TEST_FILE = os.path.join(DATA_DIR, "test_raw.csv")
MODEL_DIR = os.path.join(BASE_DIR, "models")
train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)
TARGET = "loan_paid_back"
NUMS = ["debt_to_income_ratio", "credit_score", "interest_rate", "annual_income", "loan_amount"]
CATS = ["grade_subgrade", "employment_status", "gender", "marital_status", "loan_purpose", "education_level"]
FEATURES = NUMS + CATS
train.nunique()

id                      593994
annual_income           119728
debt_to_income_ratio       526
credit_score               399
loan_amount             111570
interest_rate             1454
gender                       3
marital_status               4
education_level              5
employment_status            5
loan_purpose                 8
grade_subgrade              30
loan_paid_back               2
dtype: int64

In [686]:
og_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(), make_column_selector(dtype_include=[np.number, object])),
    ],
    remainder="passthrough",
)

In [687]:
len_train = len(train)
X = train.drop(columns=['id', TARGET])
y = train[TARGET]
X_prep = og_preprocessor.fit_transform(X, y)

In [688]:
# Designed A/B test function to check which features are actually needed
# Compares baseline vs modified feature sets → ΔAUC + std error
# 5-fold CV, TargetEncoding, XGBoost(GPU+early_stopping)
# Usage: ab_test_stat(model_template, X_baseline, X_tested_features, y)

import numpy as np


param_grid = {'max_depth': 4, 'subsample': 0.9995401032612683,
              'colsample_bytree': 0.47589024507597905, 'colsample_bynode': 0.8839133694245677,
              'reg_alpha': 1.7524064845511668e-06, 'reg_lambda': 3.8559591190677964}

model_template = XGBClassifier(
    **param_grid,
    n_estimators=10000,
    learning_rate=0.01,
    objective="binary:logistic",
    eval_metric="auc",
    early_stopping_rounds=200,
    max_bin=1024,
    random_state=42,
    enable_categorical=True,
    device="cuda",
    n_jobs=-1,
)


def ab_test_stat(model_template, X_raw, X_changed, y, cv_fold=5):
    """
    X_raw, X_changed: np.ndarray lub DataFrame
    y: Series / np.ndarray
    """
    X_raw = np.asarray(X_raw)
    X_changed = np.asarray(X_changed)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=cv_fold, shuffle=True, random_state=42)

    baseline_oof = np.zeros(len(y))
    changed_oof = np.zeros(len(y))

    for idx, (trn_idx, val_idx) in enumerate(cv.split(X_raw, y)):
        print(f"\n=== ab_test fold {idx + 1}/{cv_fold} ===")

        # 1.1. X_raw vs X_changed
        X_train_raw = X_raw[trn_idx]
        X_val_raw = X_raw[val_idx]
        X_train_changed = X_changed[trn_idx]
        X_val_changed = X_changed[val_idx]
        y_train = y[trn_idx]
        y_val = y[val_idx]

        # 1.2. TargetEncoder na danych wejściowych
        te_raw = TargetEncoder(cv=5, random_state=42)
        te_changed = TargetEncoder(cv=5, random_state=42)

        X_train_raw_enc = te_raw.fit_transform(X_train_raw, y_train)
        X_val_raw_enc = te_raw.transform(X_val_raw)

        X_train_changed_enc = te_changed.fit_transform(X_train_changed, y_train)
        X_val_changed_enc = te_changed.transform(X_val_changed)

        # 1.3. XGBoost z early stopping
        model_raw = XGBClassifier(**model_template.get_params()).fit(
            X_train_raw_enc,
            y_train,
            eval_set=[(X_val_raw_enc, y_val)],
            verbose=False,
        )
        baseline_oof[val_idx] = model_raw.predict_proba(X_val_raw_enc)[:, 1]

        model_changed = XGBClassifier(**model_template.get_params()).fit(
            X_train_changed_enc,
            y_train,
            eval_set=[(X_val_changed_enc, y_val)],
            verbose=False,
        )
        changed_oof[val_idx] = model_changed.predict_proba(X_val_changed_enc)[:, 1]

    # 2. Oblicz metryki
    baseline_scores = []
    changed_scores = []

    for idx, (trn_idx, val_idx) in enumerate(cv.split(X_raw, y)):
        auc_raw = roc_auc_score(y[val_idx], baseline_oof[val_idx])
        auc_changed = roc_auc_score(y[val_idx], changed_oof[val_idx])
        baseline_scores.append(auc_raw)
        changed_scores.append(auc_changed)

    baseline_mean = np.mean(baseline_scores)
    changed_mean = np.mean(changed_scores)
    delta = changed_mean - baseline_mean
    baseline_std = np.std(baseline_scores)
    se = np.sqrt((baseline_std ** 2 + np.std(changed_scores) ** 2) / 2)

    print(f"Baseline:  {baseline_mean:.5f} ±{baseline_std:.5f}")
    print(f"Changed:   {changed_mean:.5f} ±{np.std(changed_scores):.5f}")
    print(f"ΔAUC:      {delta:+.5f}")
    print(f"Std Error: {se:.5f}")

    return None

In [690]:
# Test rounded features: does rounding income/loan_amount add signal?
# Baseline: original continuous features
# Modified: original + rounded versions (round(0), round(-1))
def add_round_features(X):
    X_out = X.copy()
    for col in X.columns:
        X_out[f'{col}_round0'] = X[col].round(0)
        X_out[f'{col}_round10'] = X[col].round(-1)
    return X_out

add_round_transformer = FunctionTransformer(
    func=add_round_features,
    validate=False,
)

new_preprocessor = ColumnTransformer(
    transformers=[
        ("rounds", add_round_transformer, ['annual_income', 'loan_amount']),
        ("ohe", TargetEncoder(), CATS),
    ],
    remainder="passthrough",
)
X_2 = X.copy()
X_2 = new_preprocessor.fit_transform(X, y)
# ab_test_stat(model_template, X_prep, X_2, y, cv_fold=5)

In [ ]:
# Test quantile binning: equal-frequency bins capture skewed distributions better
# XGBoost learns monotonic patterns + handles outliers via stable bin counts
# Baseline: previous preprocessor (rounds + target encoding)
# Modified: + quantile bins for all NUMS → _{c}_bin5
# IDEA from Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow:
def create_quantile_bins(df, q=5):
    """Tworzy binning features"""
    bin_features = pd.DataFrame(index=df.index)
    for c in NUMS:
        try:
            train_bins, bins = pd.qcut(df[c], q=q, labels=False, retbins=True, duplicates="drop")
            bin_features[f"{c}_bin{q}"] = train_bins
        except Exception:
            bin_features[f"{c}_bin{q}"] = 0
    df = pd.concat([df, bin_features], axis=1)
    return df

quantile_bin_transformer = FunctionTransformer(
    func=create_quantile_bins,
    validate=False
)

In [ ]:
# Test uniform vs quantile binning: XGBoost internal binning works better with uniform widths
# Quantile bins = equal counts → crowded splits, uniform = equal ranges → cleaner splits
# Baseline: quantile bins (equal freq)
# Modified: uniform bins (equal width) → _{c}_ubin5
# IDEA from Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow:
def create_uniform_bins(df, q=5):
    bin_features = pd.DataFrame(index=df.index)
    for c in NUMS:
        try:
            train_bins, bins = pd.cut(
                df[c],
                bins=q,
                labels=False,
                retbins=True,
                duplicates="drop",
            )
            bin_features[f"{c}_ubin{q}"] = train_bins
        except Exception:
            bin_features[f"{c}_ubin{q}"] = 0
    df = pd.concat([df, bin_features], axis=1)
    return df


uniform_bin_transformer = FunctionTransformer(
    func=lambda X: create_uniform_bins(X, q=5),  # q możesz też przekazać przez `kw_args`
    validate=False,
)

new_preprocessor = ColumnTransformer(
    transformers=[
        ("rounds", add_round_transformer, ['annual_income', 'loan_amount']),
        ('ohe', TargetEncoder(), CATS),
        ("bins", quantile_bin_transformer, NUMS),
    ],
    remainder='passthrough'
)

In [ ]:
# Test custom interactions: domain-specific features for cleaner XGBoost splits
# grade_subgrade→numeric + credit/grade×DTI ratios = explicit risk signals
# Baseline: binned/rounded features only
# Modified: + base_grade, grade_score, score_dti_product, grade_dti_product

class FullFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        grade_map = {'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5}
        X["base_grade"] = X["grade_subgrade"].str[0]
        X["grade_score"] = X["base_grade"].map(grade_map)
        X['income_grade_interaction'] = X['annual_income'] * X['grade_score']

        return X

In [ ]:
# Test smart polynomial features: XGBoost captures interactions better with explicit terms
# Trees need fewer splits for dti*credit vs learning it implicitly through splits
# Baseline: bins/rounds only
# Modified: + dti², credit², income², dti*credit, dti*income, credit*income
class SmartPolyFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, degree=2):
        self.degree = degree
        self.top_features = ['debt_to_income_ratio', 'credit_score', 'annual_income']
        self.feature_names = None

    def fit(self, X, y=None):
        available_features = [col for col in self.top_features if col in X.columns]
        self.available_features = available_features
        return self

    def transform(self, X):
        X_poly = X.copy()

        if self.available_features:
            dti, credit, income = self.available_features

            X_poly['dti_squared'] = X[dti] ** 2
            if 'credit_score' in X.columns:
                X_poly['credit_squared'] = X['credit_score'] ** 2
            if 'annual_income' in X.columns:
                X_poly['income_squared'] = X['annual_income'] ** 2

            X_poly['dti_credit'] = X[dti] * X['credit_score']
            X_poly['dti_income'] = X[dti] * X['annual_income']
            X_poly['credit_income'] = X['credit_score'] * X['annual_income']

        return X_poly

In [ ]:
# Test digit extraction: XGBoost loves discrete integers over continuous floats
# Trees split better on exact digits (annual_income_d-4=3 vs 3.1) → cleaner splits
# Baseline: poly features + bins/rounds
# Modified: + digit features e.g. annual_income_d-4, loan_amount_d0, interest_rate_d1
class DigitFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns or ['annual_income', 'loan_amount', 'interest_rate', 'debt_to_income_ratio',
                                   'credit_score']
        self.digit_features = []

    def fit(self, X, y=None):
        self.digit_features = []
        return self

    def transform(self, X):
        X_out = X.copy()

        configs = {
            'annual_income': [(-4, '_d-4'), (-3, '_d-3'), (-2, '_d-2'), (-1, '_d-1'), (0, '_d0'), (1, '_d1')],
            'loan_amount': [(-4, '_d-4'), (-3, '_d-3'), (-2, '_d-2'), (-1, '_d-1'), (0, '_d0'), (1, '_d1')],
            'interest_rate': [(-1, '_d-1'), (0, '_d0'), (1, '_d1'), (2, '_d2')],
            'debt_to_income_ratio': [(1, '_d1'), (2, '_d2'), (3, '_d3')],
            'credit_score': [(0, '_d0'), (1, '_d1'), (2, '_d2')]
        }

        for col in self.columns:
            if col in X.columns and col in configs:
                for power, suffix in configs[col]:
                    new_col = f'{col}{suffix}'

                    digit = ((X[col] * (10 ** power)) % 10).fillna(-1).astype("int8")
                    X_out[new_col] = digit
                    self.digit_features.append(new_col)

        return X_out

In [ ]:
high_cardinality = [
    "annual_income",
    "loan_amount",
]

new_preprocessor = ColumnTransformer(
    transformers=[
        ('te', TargetEncoder(cv=5, random_state=42, shuffle=True),
         make_column_selector(dtype_include=[object])),
        ("count", CountEncoder(), make_column_selector(dtype_include=[object])),
        ('te_nums', TargetEncoder(cv=5, random_state=42, shuffle=True),
         NUMS),
        ("rounds", add_round_transformer, high_cardinality),
        ("bins", quantile_bin_transformer, high_cardinality),
        ("bins_uniform", uniform_bin_transformer, high_cardinality)

    ],
    remainder='passthrough'
)
pipeline = Pipeline([
    ('fe', FullFeatureEngineering()),
    ('digits', DigitFeaturesTransformer()),
    # ('poly', SmartPolyFeatures()),
    ('new', new_preprocessor)
])
X_2 = X.copy()
X_final = pipeline.fit_transform(X_2, y)
test_id = test['id']
test_processed = test.drop(columns=['id']).copy()
X_test_prep = pipeline.transform(test_processed)

n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

param_grid = {'max_depth': 6, 'subsample': 0.6983311354996038, 'colsample_bytree': 0.257941,
              'colsample_bynode': 0.870405260085986, 'reg_alpha': 5.530877836114476e-06,
              'reg_lambda': 4.907938785602363e-05, "gamma":0.03}

model_template = XGBClassifier(
    **param_grid,
    n_estimators=10000, # This one here saved the entire run
    learning_rate=0.01,
    objective="binary:logistic",
    eval_metric="auc",
    early_stopping_rounds=500,
    random_state=42,
    bin_size=1500,
    enable_categorical=True,
    device="cuda",
    n_jobs=-1,
)

out_of_fold_predictions = np.zeros(len(y))
predictions = np.zeros(len(test))
cv_scores = []

for idx, (trn_idx, val_idx) in enumerate(skf.split(X_final, y)):
    print(f"\n=== Fold {idx + 1}/{n_splits} ===")

    X_train = X_final[trn_idx]
    X_val = X_final[val_idx]
    X_test_enc = X_test_prep.copy()
    y_train = y.iloc[trn_idx]
    y_val = y.iloc[val_idx]

    model = XGBClassifier(**model_template.get_params()).fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_proba = model.predict_proba(X_val)[:, 1]
    out_of_fold_predictions[val_idx] = val_proba
    predictions += model.predict_proba(X_test_enc)[:, 1]

    auc = roc_auc_score(y_val, val_proba)
    cv_scores.append(auc)
    print(f"Fold {idx + 1} AUC: {auc:.6f}")

predictions /= n_splits
print(f"\nFinal CV AUC: {roc_auc_score(y, out_of_fold_predictions):.6f}")

submission = pd.DataFrame({"id": test_id, "loan_paid_back": predictions})
submission_path = os.path.join(MODEL_DIR, "submission_that_is_blessed_by_Holy.csv")
submission.to_csv(submission_path, index=False)

In [ ]:
# Optuna hyperparameter tuning for XGBoost (1000 trials)
# Found optimal params for GPU-accelerated model with TargetEncoder baseline
# Saved to best_xgb_params.txt → used in ab_test_stat model_template
# from sklearn.preprocessing import TargetEncoder
# from sklearn.metrics import roc_auc_score
# import warnings
#
# warnings.filterwarnings("ignore")
# import optuna
# from xgboost import XGBClassifier
# from sklearn.model_selection import cross_val_score
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('te', TargetEncoder(), CATS)
#     ],
#     remainder='passthrough'
# )
# X = preprocessor.fit_transform(X, y)
# X_tr, X_val, y_tr, y_val = train_test_split(X, y, random_state=42, test_size=0.2)
# def objective(trial):
#     params = {
#         "max_depth": trial.suggest_int("max_depth", 5, 6),
#         "subsample": trial.suggest_float("subsample", 0.95, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.5),
#         "colsample_bynode": trial.suggest_float("colsample_bynode", 0.75, 0.95),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1e-4, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
#         "device": "cuda",
#         "tree_method": "hist",
#         "predictor": "gpu_predictor",
#         "objective": "binary:logistic",
#         "eval_metric": "auc",
#         "random_state": 42,
#         "n_jobs": -1
#     }
#
#     model = XGBClassifier(**params, early_stopping_rounds=500, learning_rate=0.01, n_estimators=10000)
#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_val, y_val)],
#         verbose=False
#     )
#
#     val_proba = model.predict_proba(X_val)[:, 1]
#     return roc_auc_score(y_val, val_proba)
#
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=1000)
# best_params = study.best_params
# print(best_params)
# with open("best_xgb_params.txt", "w") as f:
#     for key, value in best_params.items():
#         f.write(f"{key} = {value}\n")

In [ ]:
# Huge thanks to my partner in crime Viyaleta Reut who saved me with EDA Documentation and AI Subscription